# LSTM dự báo nhu cầu thuê xe theo thời gian

Notebook sử dụng dữ liệu đã tiền xử lý trong `data/processed`, giữ thứ tự thời gian và không shuffle khi huấn luyện.

- Target: `cnt`
- Cửa sổ thời gian: 24 quan sát trước đó
- Scaler chỉ fit trên tập train
- Đánh giá: RMSE, MAE, R²

In [1]:
from pathlib import Path
from time import perf_counter
import os
import random

# Buộc TensorFlow chạy trên CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.set_visible_devices([], "GPU")
except RuntimeError:
    pass

project_root = Path.cwd()
if not (project_root / "data/processed/train.csv").exists():
    project_root = project_root.parent

processed_dir = project_root / "data/processed"
metrics_dir = project_root / "results/metrics"
models_dir = project_root / "models"
metrics_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root.resolve()}")
print(f"TensorFlow: {tf.__version__}")

Project root: E:\bike-demand-prediction-main
TensorFlow: 2.20.0


In [2]:
def load_split(name):
    return pd.read_csv(
        processed_dir / f"{name}.csv",
        parse_dates=["timestamp", "dteday"],
    ).sort_values("timestamp").reset_index(drop=True)

train_df = load_split("train")
validation_df = load_split("validation")
test_df = load_split("test")

target_column = "cnt"
excluded_columns = {
    target_column,
    "instant",
    "dteday",
    "timestamp",
    "casual",
    "registered",
}
feature_columns = [
    column for column in train_df.columns if column not in excluded_columns
]
sequence_length = 24

assert train_df["timestamp"].max() < validation_df["timestamp"].min()
assert validation_df["timestamp"].max() < test_df["timestamp"].min()
assert train_df[feature_columns].notna().all().all()
assert validation_df[feature_columns].notna().all().all()
assert test_df[feature_columns].notna().all().all()

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(train_df[feature_columns])
X_validation_scaled = feature_scaler.transform(validation_df[feature_columns])
X_test_scaled = feature_scaler.transform(test_df[feature_columns])

y_train_scaled = target_scaler.fit_transform(train_df[[target_column]]).ravel()
y_validation_scaled = target_scaler.transform(validation_df[[target_column]]).ravel()
y_test_scaled = target_scaler.transform(test_df[[target_column]]).ravel()

def make_sequences(features, targets, window):
    sequence_features = []
    sequence_targets = []
    for index in range(window, len(features)):
        sequence_features.append(features[index - window:index])
        sequence_targets.append(targets[index])
    return np.asarray(sequence_features), np.asarray(sequence_targets)

X_train_seq, y_train_seq = make_sequences(
    X_train_scaled, y_train_scaled, sequence_length
)
validation_context_X = np.vstack(
    [X_train_scaled[-sequence_length:], X_validation_scaled]
)
validation_context_y = np.concatenate(
    [y_train_scaled[-sequence_length:], y_validation_scaled]
)
X_validation_seq, y_validation_seq = make_sequences(
    validation_context_X, validation_context_y, sequence_length
)
test_context_X = np.vstack([X_validation_scaled[-sequence_length:], X_test_scaled])
test_context_y = np.concatenate([y_validation_scaled[-sequence_length:], y_test_scaled])
X_test_seq, y_test_seq = make_sequences(
    test_context_X, test_context_y, sequence_length
)

assert len(X_train_seq) == len(train_df) - sequence_length
assert len(X_validation_seq) == len(validation_df)
assert len(X_test_seq) == len(test_df)

print(f"Features: {len(feature_columns)}")
print(f"Sequences: train={X_train_seq.shape}, validation={X_validation_seq.shape}, test={X_test_seq.shape}")

Features: 27
Sequences: train=(12023, 24, 27), validation=(2582, 24, 27), test=(2582, 24, 27)


In [3]:
lstm_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(sequence_length, len(feature_columns))
        ),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1),
    ],
    name="bike_demand_lstm",
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

start_time = perf_counter()
history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_validation_seq, y_validation_seq),
    epochs=30,
    batch_size=64,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1,
)
training_time = perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")
print(f"Epochs completed: {len(history.history['loss'])}")

Epoch 1/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.4839 - val_loss: 0.6972
Epoch 2/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2243 - val_loss: 0.4110
Epoch 3/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1414 - val_loss: 0.2974
Epoch 4/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1063 - val_loss: 0.2357
Epoch 5/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0872 - val_loss: 0.2167
Epoch 6/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0765 - val_loss: 0.2094
Epoch 7/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0696 - val_loss: 0.2078
Epoch 8/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0644 - val_loss: 0.2104
Epoch 9/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0604 - val_loss: 0.2140
Epoch 10/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0574 - val_loss: 0.2189
Epoch 11/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0550 - val_loss: 0.2260
Training time: 11.53 seconds
Epochs completed: 11


In [ ]:
def evaluate_split(split_name, y_true_scaled, X_sequence):
    predictions_scaled = lstm_model.predict(X_sequence, verbose=0).ravel()
    y_true = target_scaler.inverse_transform(
        y_true_scaled.reshape(-1, 1)
    ).ravel()
    predictions = target_scaler.inverse_transform(
        predictions_scaled.reshape(-1, 1)
    ).ravel()
    return {
        "model": "LSTM",
        "split": split_name,
        "RMSE": mean_squared_error(y_true, predictions) ** 0.5,
        "MAE": mean_absolute_error(y_true, predictions),
        "R2": r2_score(y_true, predictions),
        "Training Time": training_time,
    }

lstm_metrics = pd.DataFrame(
    [
        evaluate_split("validation", y_validation_seq, X_validation_seq),
        evaluate_split("test", y_test_seq, X_test_seq),
    ]
)

print("LSTM results:")
display(lstm_metrics.round(4))
lstm_metrics.to_csv(metrics_dir / "lstm_metrics.csv", index=False)

lstm_model.save(models_dir / "lstm.keras")
print(f"Saved model to: {models_dir / 'lstm.keras'}")

LSTM results:


,model,split,RMSE,MAE,R2,Training Time
0,LSTM,validation,69.9274,50.8008,0.9028,11.5304
1,LSTM,test,84.4831,59.5104,0.8441,11.5304


Saved model to: e:\bike-demand-prediction-main\models\lstm.keras


: 